In [5]:
PATH = "/home/jmurga/mkt/202004/"
library(data.table)
library(dplyr)
source(paste0(PATH,"scripts/src/postprocessing.R"))

# Convert SFS to DOFE

Please first create the folder to perform the analysis
```bash
polyPath=/home/jmurga/mkt/202004/
mkdir -p ${polyPath}/rawData/dofe/polydfe
```

## noDemog

In [2]:
df= fread(paste0(PATH,"rawData/simulations/bgsTable.tsv"))

In [ ]:
analysis = c("noDemog_0.4_0.1_0.2","noDemog_0.4_0.1_0.4","noDemog_0.4_0.1_0.8","noDemog_0.4_0.1_0.999","noDemog_0.4_0.2_0.2","noDemog_0.4_0.2_0.4","noDemog_0.4_0.2_0.8","noDemog_0.4_0.2_0.999","noDemog_0.4_0.3_0.2","noDemog_0.4_0.3_0.4","noDemog_0.4_0.3_0.8","noDemog_0.4_0.3_0.999")

In [ ]:
alphas = list()
for(n in analysis){
    print(n)
    tmp = unlist(strsplit(n,'_'))
    B = tmp[4]; alphaW = tmp[3]
    
    for(i in 1:100){
        sfsFile = paste0(PATH,"rawData/dofe/noDemog/",n,"/",n,"_polydfe_",i,"_10.tsv")

        outDiv = paste0(PATH,"results/polydfe/noDemog/",n,"/",n,"_alphadiv_",i,".txt")

        outDfe = paste0(PATH,"results/polydfe/noDemog/",n,"/",n,"_alphadfe_",i,".txt")

        est_div = parseOutput(outDiv)
        est_dfe = parseOutput(outDiv)

        div = parseDivergenceData(sfsFile)
        
        alpha_div = round(sapply(est_div[1], function(e) c("supLimit = 0" = estimateAlpha(e,div=div, poly = FALSE),"supLimit = 10" = estimateAlpha(e, supLimit = 10, div=div,poly = FALSE))),3)
       
         alpha_dfe = round(sapply(est_dfe[1], function(e) c("supLimit = 0" = estimateAlpha(e, poly = FALSE),"supLimit = 10" = estimateAlpha(e, supLimit = 10, poly = FALSE))),3)
              
        tmp1 = c(n,alpha_div[1]-alpha_div[2],alpha_div[2],alpha_div[1],B,alphaW,est_div[[1]]$criteria,'div') %>% t
        tmp2 = c(n,alpha_dfe[1]-alpha_dfe[2],alpha_dfe[2],alpha_dfe[1],B,alphaW,est_dfe[[1]]$criteria,'dfe') %>% t

        tmp = rbind(tmp1,tmp2)
        alphas[[paste0(n,"_",i)]] = as.data.table(tmp)
    }
}
                   
alphas = rbindlist(alphas)
names(alphas) = c('analysis','alphaWeak','alphaStrong','alpha','B','alphaW','gradient','dfe')

## Isolation

In [13]:
df= fread(paste0(PATH,"rawData/simulations/tennesen.tsv"))
df$B = round(df$B,3)

In [14]:
df[['path']] = apply(df, 1 , function(x) paste0(PATH,"/rawData/simulations/isolation/isolation_",x[5],"_",x[4],"_",x[7]))

In [15]:
df

bgsThetaF,pposL,pposH,alphaW,alpha,estimation,B,path
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
9.0407e-08,0.003859879,2.393394e-04,0.1,0.4,0.25688,0.200,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.1_0.2
9.0407e-08,0.007705195,1.592586e-04,0.2,0.4,0.23086,0.200,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.2_0.2
9.0407e-08,0.011536030,7.947935e-05,0.3,0.4,0.20285,0.200,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.3_0.2
5.1471e-08,0.003859879,2.393394e-04,0.1,0.4,0.31311,0.400,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.1_0.4
5.1471e-08,0.007705195,1.592586e-04,0.2,0.4,0.29542,0.400,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.2_0.4
5.1471e-08,0.011536030,7.947935e-05,0.3,0.4,0.27673,0.400,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.3_0.4
1.2535e-08,0.003859879,2.393394e-04,0.1,0.4,0.37851,0.800,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.1_0.8
1.2535e-08,0.007705195,1.592586e-04,0.2,0.4,0.37433,0.800,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.2_0.8
1.2535e-08,0.011536030,7.947935e-05,0.3,0.4,0.37008,0.800,/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.3_0.8


In [18]:
for(n in 1:nrow()){
    sfsToPolyDFE(path=df[n]$path,bins=20,output="/home/jmurga/mkt/202004/rawData/dofe/polydfe/inputs/isolation/")
}

ERROR: Error in fread(paste0(path, "/sfs.tsv"), sep = "\t"): File '/home/jmurga/mkt/202004//rawData/simulations/isolation/isolation_0.4_0.1_0.2/sfs.tsv' does not exist or is non-readable. getwd()=='/home/jmurga/mkt/202004/scripts/notebooks'


In [31]:
sfsToPolyDFE(path=df[7]$path,bins=20,output="/home/jmurga/mkt/202004/rawData/dofe/polydfe/inputs/isolation/")

In [ ]:
analysis = c("isolation_0.4_0.1_0.2","isolation_0.4_0.1_0.8","isolation_0.4_0.1_0.999","isolation_0.4_0.2_0.2","isolation_0.4_0.2_0.8","isolation_0.4_0.2_0.999","isolation_0.4_0.3_0.2","isolation_0.4_0.3_0.8","isolation_0.4_0.3_0.999")

In [ ]:
alphas = list()
for(n in analysis){
    
    print(n)
    tmp = unlist(strsplit(n,'_'))
    B = tmp[4]; alphaW = tmp[3]
    
    sfsFile = paste0(PATH,"rawData/dofe/polydfe/inputs/isolation/",n,"_polydfe_20.tsv")
    outFile = paste0(PATH,"rawData/dofe/polydfe/outputs/isolation/r_i/",n,".polydfe")
    
    est = parseOutput(outFile)
    div = parseDivergenceData(sfsFile)
    
    alpha = round(sapply(est[1], function(e) c("supLimit = 0" = estimateAlpha(e,div=div),"supLimit = 10" = estimateAlpha(e, supLimit = 10,div=div))),3)
    alpha = c(n,alpha[2],alpha[1]-alpha[2],alpha[1],B,alphaW,est[[1]]$criteria)
    alphas[[n]] = as.data.table(t(alpha))
    
plotEstimatedSFS(sfsFile=paste0(PATH,"rawData/dofe/polydfe/inputs/isolation/",n,"_polydfe_20.tsv"),outFile=paste0(PATH,"rawData/dofe/polydfe/outputs/isolation/r_i/",n,".polydfe"),output=paste0(PATH,"results/polydfe/isolation/",n,".svg"))
    
}
                   
alphas = rbindlist(alphas)
names(alphas) = c('analysis','alphaWeak','alphaStrong','alpha','B','alphaW','gradient')